# NeuroVision v3 — Baseline-normalized SANS relationship explorer
Run install cell, then app cell. Built-in data are synthetic. CFD/FEA remain simplified teaching surrogates.

In [1]:
!pip -q install gradio pandas plotly


In [ ]:
from datetime import datetime, timezone
import math
import gradio as gr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Fictional data solely to demonstrate the analysis workflow.
# Each series is aligned to the same illustrative mission-day schedule.
RAW = pd.DataFrame({
    "Mission day": [0, 15, 30, 45, 60, 75, 90],
    "Venous-flow proxy (a.u.)": [1.00, .96, .91, .88, .86, .90, .92],
    "CSF-dynamics proxy (a.u.)": [1.00, 1.03, 1.06, 1.08, 1.09, 1.07, 1.05],
    "Peripapillary thickness (µm)": [100, 101, 103, 105, 107, 106, 104],
    "Visual-function score (a.u.)": [100, 100, 99, 98, 97, 98, 99]
})
MEASURES = list(RAW.columns[1:])
def stamp(): return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
def blank(): return {"daily": [], "research": []}

def normalized_data():
    out = RAW[["Mission day"]].copy()
    for col in MEASURES:
        baseline = RAW[col].iloc[0]
        out[col + " · baseline index"] = RAW[col] / baseline * 100 if baseline else float("nan")
    return out

NORM = normalized_data()
NORM_COLS = list(NORM.columns[1:])

def daily_save(s,b,d,f,v,n):
    s=dict(s or blank())
    s["daily"]=(s["daily"]+[{"Recorded (UTC)":stamp(),"Blurred vision":b,"Distorted vision":d,
      "Focusing difficulty":f,"Visual-field change":v,"Notes":n or ""}])[-100:]
    msg=("**Change recorded.** This tool cannot determine the cause. Follow mission medical procedures and notify the flight surgeon/medical team promptly for new, worsening, sudden or persistent visual symptoms."
         if "Yes" in [b,d,f,v] else "Check-in recorded. This is not medical clearance and does not rule out SANS.")
    return s,msg,pd.DataFrame(s["daily"])

def research_save(s,a,dom,m,val,u,side,protocol,note):
    s=dict(s or blank()); s["research"]=(s["research"]+[{"Recorded (UTC)":stamp(),"Assessment":a,"Domain":dom,
      "Measurement":m,"Value":val,"Unit":u,"Eye/side":side,"Protocol":protocol,"Notes":note}])[-200:]
    return s,"Logged. Verify units, device, protocol and clinical source before interpretation.",pd.DataFrame(s["research"])

def timeline_raw():
    # Separate panels preserve original physical units; no mixed-unit y-axis.
    fig=go.Figure()
    for col in MEASURES:
        fig.add_trace(go.Scatter(x=RAW["Mission day"],y=RAW[col],mode="lines+markers",name=col))
    fig.update_layout(title="Raw measurements — separate units shown in hover/legend",
                      xaxis_title="Mission day (shared time axis)",yaxis_title="Raw value (units differ)",
                      height=470,legend=dict(orientation="h",y=-.28))
    # Four independent y-axes would still confuse; use faceted small multiples instead.
    long=RAW.melt(id_vars="Mission day",var_name="Measurement",value_name="Raw value")
    fac=px.line(long,x="Mission day",y="Raw value",facet_row="Measurement",markers=True,
                title="Raw measurements in separate panels (preserves each unit)")
    fac.update_layout(height=780,showlegend=False)
    fac.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    return fac

def timeline_normalized():
    long=NORM.melt(id_vars="Mission day",var_name="Measurement",value_name="Baseline index")
    fig=px.line(long,x="Mission day",y="Baseline index",color="Measurement",markers=True,
                title="Baseline-normalized trends — each series starts at 100")
    fig.add_hline(y=100,line_dash="dash",annotation_text="Personal baseline = 100")
    fig.update_layout(height=470,xaxis_title="Mission day (shared time axis)",
                      yaxis_title="Index (% of baseline)",legend=dict(orientation="h",y=-.3))
    return fig

def normalized_table():
    return NORM.round(2)

def correlation_matrix():
    # Correlations calculated from normalized series; linear rescaling leaves Pearson r unchanged.
    corr=NORM[NORM_COLS].corr()
    fig=px.imshow(corr,text_auto=".2f",zmin=-1,zmax=1,aspect="auto",
                  title="Pairwise correlation — synthetic normalized series")
    fig.update_layout(height=480)
    return fig

def relationship(x,y):
    df=NORM[["Mission day",x,y]].dropna()
    r=df[x].corr(df[y])
    fig=px.scatter(df,x=x,y=y,color="Mission day",hover_data=["Mission day"],
                   title="Normalized association across aligned mission days")
    fig.add_hline(y=100,line_dash="dash")
    fig.add_vline(x=100,line_dash="dash")
    note=(f"Pearson correlation: **{r:.2f}** across {len(df)} synthetic time points. "
          "Both axes are baseline indices (100 = each measure's own baseline). "
          "This is descriptive association only—not causation, a clinical threshold, or a validated prediction.")
    return fig,note

def lag_explorer(x,y,lag):
    # Positive lag means candidate mechanism at day t is compared with outcome at later day t+lag.
    a=NORM[["Mission day",x]].copy()
    b=NORM[["Mission day",y]].copy()
    b["Mission day"]=b["Mission day"]-lag
    joined=a.merge(b,on="Mission day",suffixes=("_mechanism","_outcome")).dropna()
    if len(joined)<3:
        return f"Only {len(joined)} aligned pairs at this lag; too few for even an exploratory correlation."
    r=joined[x].corr(joined[y])
    return (f"At an illustrative lag of **{lag} mission days**, {len(joined)} paired points give r = **{r:.2f}**. "
            "This is an exploratory timing comparison in synthetic data. It does not establish biological delay or causation.")

def cfd(radius_mm,length_mm,dp_pa,mu_mpas):
    r=radius_mm/1000; L=length_mm/1000; mu=mu_mpas/1000
    q=math.pi*r**4*dp_pa/(8*mu*L); vel=q/(math.pi*r*r); shear=4*mu*q/(math.pi*r**3)
    return (f"**Idealized flow rate:** {q*1e9:.4g} µL/s  \n**Mean velocity:** {vel*1000:.4g} mm/s  \n"
            f"**Wall shear estimate:** {shear:.4g} Pa  \n\n"
            "Steady laminar Newtonian flow in a rigid circular tube—not anatomical CFD of CSF around the optic nerve. It omits annular anatomy, pulsatility, compliant walls, tissue exchange and validated boundary conditions.")

def fea(stress_kpa,E_mpa,L_mm):
    strain=(stress_kpa*1000)/(E_mpa*1e6); disp=strain*(L_mm/1000)
    return (f"**Nominal strain:** {strain:.4g} ({strain*100:.4g}%)  \n**Displacement proxy:** {disp*1e6:.4g} µm  \n\n"
            "One-dimensional homogeneous linear-elastic proxy, not an FEA mesh or optic nerve tissue model.")



import base64
from io import BytesIO
from PIL import Image

EYE_ART = Image.open(BytesIO(base64.b64decode("...")))

# Dark aerospace-medical command-center visual system.
CUSTOM_CSS = """
.gradio-container {
  max-width: 1600px !important;
  background:
    radial-gradient(ellipse at 52% 18%, rgba(0,126,190,.18), transparent 38%),
    radial-gradient(ellipse at 92% 5%, rgba(75,65,170,.15), transparent 28%),
    linear-gradient(145deg,#030914 0%,#061426 55%,#07101e 100%) !important;
  color:#e9f7ff !important;
}
.gradio-container, .gradio-container * { font-family: Inter, Arial, sans-serif; }
.gradio-container h1,.gradio-container h2,.gradio-container h3,.gradio-container p,
.gradio-container label,.gradio-container .prose { color:#e9f7ff !important; }
.nv-top {
  display:flex; align-items:center; justify-content:space-between; gap:20px;
  padding:18px 24px; margin:4px 0 14px;
  border:1px solid rgba(48,190,235,.42); border-radius:16px;
  background:linear-gradient(100deg,rgba(5,29,53,.98),rgba(5,17,34,.94));
  box-shadow:0 0 26px rgba(0,174,230,.12), inset 0 1px 0 rgba(255,255,255,.06);
}
.nv-brand {font-size:34px; font-weight:850; letter-spacing:.06em; color:#f3fbff;}
.nv-brand span {color:#4de1ff;}
.nv-tag {font-size:11px; letter-spacing:.2em; color:#61dafa; text-transform:uppercase; font-weight:800;}
.nv-sub {font-size:14px; color:#b9d8ea; margin-top:4px;}
.nv-chip {display:inline-block; border:1px solid rgba(72,212,247,.38); color:#aeeeff;
  padding:6px 10px; border-radius:30px; font-size:10px; letter-spacing:.08em; margin:3px;}
.nv-panel {
  background:linear-gradient(155deg,rgba(9,31,53,.97),rgba(5,18,34,.96)) !important;
  border:1px solid rgba(48,177,222,.42) !important; border-radius:14px !important;
  box-shadow:0 8px 26px rgba(0,0,0,.24), inset 0 1px 0 rgba(255,255,255,.035);
  padding:13px !important; min-height:100%;
}
.nv-panel h3 {color:#8feaff !important; font-size:14px !important; letter-spacing:.08em; text-transform:uppercase;}
.nv-small {font-size:11px; color:#94b6cb; letter-spacing:.05em;}
.nv-stat {font-size:24px; font-weight:800; color:#52e3ff; margin:7px 0 2px;}
.nv-status {display:flex; justify-content:space-between; border-bottom:1px solid rgba(84,160,194,.17);
  padding:8px 0; font-size:12px; color:#d7edf8;}
.nv-on {color:#55e5b0; font-weight:700;}
.nv-plan {color:#c5a8ff; font-weight:700;}
.nv-goal {
  border:1px solid rgba(177,120,255,.6); border-radius:12px; padding:12px;
  background:linear-gradient(90deg,rgba(76,38,128,.32),rgba(13,29,58,.9));
  text-align:center; color:#eadbff; font-weight:800; letter-spacing:.04em;
  box-shadow:0 0 20px rgba(151,93,255,.13);
}
.nv-disclosure {
  border-left:3px solid #a98aff; border-radius:7px; padding:9px 12px;
  background:rgba(93,67,145,.16); color:#d9d2ed; font-size:11px;
}
.nv-section {color:#54dcf8 !important; font-size:11px !important; font-weight:800;
  letter-spacing:.14em; text-transform:uppercase;}
.gradio-container .tabs {border-bottom:1px solid rgba(48,177,222,.35) !important;}
.gradio-container button[role=tab] {color:#a9c8db !important; background:transparent !important;}
.gradio-container button[role=tab][aria-selected=true] {
  color:#061421 !important; background:linear-gradient(90deg,#40d9f5,#82edff) !important;
  border-radius:8px 8px 0 0 !important; font-weight:800 !important;
}
.gradio-container .block,.gradio-container .form,.gradio-container .panel {
  background:rgba(8,25,44,.92) !important; border:1px solid rgba(48,177,222,.25) !important;
  border-radius:11px !important;
}
.gradio-container input,.gradio-container textarea,.gradio-container select {
  background:#071a2d !important; color:#eaf8ff !important; border-color:rgba(71,177,220,.35) !important;
}
.gradio-container button.primary {
  background:linear-gradient(95deg,#087eae,#22c5df) !important; color:#04121e !important;
  font-weight:800 !important; border:0 !important;
}
.gradio-container .plot-container {border-radius:10px; overflow:hidden;}
.nv-foot {border-top:1px solid rgba(48,177,222,.25); padding-top:10px; color:#87a9bf; font-size:10px; letter-spacing:.08em;}
"""

# Plotly charts use the same visual palette as the command-center UI.
DARK_LAYOUT = dict(
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(3,14,27,.35)",
    font=dict(color="#cfe9f5", family="Inter, Arial"),
    margin=dict(l=42,r=18,t=42,b=38),
    xaxis=dict(gridcolor="rgba(90,160,190,.16)", zerolinecolor="rgba(90,160,190,.18)"),
    yaxis=dict(gridcolor="rgba(90,160,190,.16)", zerolinecolor="rgba(90,160,190,.18)")
)
def dashboard_trend():
    fig=timeline_normalized()
    fig.update_layout(**DARK_LAYOUT, height=245, title="Illustrative baseline-normalized trends",
                      legend=dict(orientation="h",y=-.28,font=dict(size=9)))
    fig.update_traces(line=dict(width=2.5), marker=dict(size=6))
    return fig

with gr.Blocks(
    title="NEUROVISION | Spaceflight Vision Research",
    css=CUSTOM_CSS,
    theme=gr.themes.Base(primary_hue="cyan", secondary_hue="blue", neutral_hue="slate")
) as app:
    state=gr.State(blank())

    gr.HTML("""
    <div class="nv-top">
      <div>
        <div class="nv-tag">Spaceflight vision research platform</div>
        <div class="nv-brand">NEURO<span>VISION</span></div>
        <div class="nv-sub">From ocular monitoring to optic nerve protection and regeneration research</div>
      </div>
      <div style="text-align:right">
        <span class="nv-chip">VISION MONITORING</span>
        <span class="nv-chip">CFD / FEA</span>
        <span class="nv-chip">FUTURE SI AGENT</span>
        <div class="nv-small" style="margin-top:8px">RESEARCH PROTOTYPE · NOT FOR CLINICAL DECISIONS</div>
      </div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=3, min_width=220):
            with gr.Group(elem_classes=["nv-panel"]):
                gr.Markdown("### ◉ VISION MONITORING")
                gr.Markdown("<div class='nv-small'>Ocular imaging & visual function</div>")
                gr.Markdown("**Research domains**")
                gr.Markdown("- OCT / RNFL structure\n- Visual function\n- Ocular imaging\n- Longitudinal observations")
                gr.Markdown("<div class='nv-stat'>4</div><div class='nv-small'>INTERACTIVE RESEARCH WORKSPACES</div>")
                gr.Markdown("<div class='nv-disclosure'>Measurement records can be entered with units, eye/side, instrument and protocol.</div>")

        with gr.Column(scale=6, min_width=360):
            with gr.Group(elem_classes=["nv-panel"]):
                gr.Markdown("<div class='nv-section'>CENTRAL ANATOMY VIEW · CONCEPTUAL ILLUSTRATION</div>")
                gr.Image(value=EYE_ART, show_label=False, interactive=False, height=300, container=False)
                gr.HTML("""
                <div class="nv-goal">
                  ✧ LONG-TERM RESEARCH AMBITION<br>
                  <span style="font-size:17px;color:#ffffff">Vision restoration through optic nerve regeneration research</span>
                </div>
                """)

        with gr.Column(scale=3, min_width=220):
            with gr.Group(elem_classes=["nv-panel"]):
                gr.Markdown("### ◷ RESEARCH STATUS")
                gr.HTML("""
                <div class="nv-status"><span>Daily observations</span><span class="nv-on">Available</span></div>
                <div class="nv-status"><span>Measurement logging</span><span class="nv-on">Available</span></div>
                <div class="nv-status"><span>Trend exploration</span><span class="nv-on">Available</span></div>
                <div class="nv-status"><span>CFD / FEA proxies</span><span class="nv-on">Available</span></div>
                <div class="nv-status"><span>NASA data integration</span><span class="nv-plan">Planned</span></div>
                <div class="nv-status"><span>SI research agent</span><span class="nv-plan">Planned</span></div>
                <div class="nv-status"><span>Regeneration research</span><span class="nv-plan">Long-term</span></div>
                """)
                gr.Markdown("### ◈ DATA PROVENANCE")
                gr.Markdown("<div class='nv-small'>Built-in example records: synthetic<br>Manual records: user-entered<br>NASA / device connections: not active</div>")

    with gr.Row(equal_height=True):
        with gr.Column(scale=4):
            with gr.Group(elem_classes=["nv-panel"]):
                gr.Markdown("### ≋ CFD · FLUID MECHANICS")
                gr.Markdown("<div class='nv-small'>Idealized flow and pressure calculations</div>")
                gr.Markdown("**Explore:** flow rate · mean velocity · wall shear estimate")
                gr.Markdown("<div class='nv-small'>Current implementation: rigid circular-tube analytical proxy, not anatomical CSF CFD.</div>")
        with gr.Column(scale=4):
            with gr.Group(elem_classes=["nv-panel"]):
                gr.Markdown("### ◈ FEA · TISSUE MECHANICS")
                gr.Markdown("<div class='nv-small'>Simplified stress and deformation exploration</div>")
                gr.Markdown("**Explore:** nominal strain · displacement proxy")
                gr.Markdown("<div class='nv-small'>Current implementation: one-dimensional linear-elastic proxy, not an anatomical FEA model.</div>")
        with gr.Column(scale=5):
            with gr.Group(elem_classes=["nv-panel"]):
                gr.Markdown("### ▥ LONGITUDINAL TRENDS")
                gr.Markdown("<div class='nv-small'>Synthetic example data · each series normalized to its own baseline</div>")
                gr.Plot(value=dashboard_trend())

    gr.Markdown("""
    <div class="nv-disclosure">
    <b>Scientific disclosure:</b> Built-in records are synthetic demonstration data. Manual entries are not connected to NASA systems or medical devices. The CFD/FEA workbench uses simplified mathematical proxies. NeuroVision is not a diagnostic, clinical decision-support, or validated prediction tool.
    </div>
    """)

    gr.Markdown("## RESEARCH WORKSPACES")
    with gr.Tabs():
        with gr.Tab("01 · Daily check-in"):
            gr.Markdown("Record SANS-relevant visual symptoms. Report new or worsening symptoms under mission medical procedures.")
            with gr.Row():
                b=gr.Radio(["No","Yes","Unsure"],label="Blurred/reduced visual clarity?")
                d=gr.Radio(["No","Yes","Unsure"],label="Distorted/wavy vision?")
            with gr.Row():
                f=gr.Radio(["No","Yes","Unsure"],label="New focusing difficulty?")
                v=gr.Radio(["No","Yes","Unsure"],label="New visual-field change/blind spot?")
            n=gr.Textbox(label="Optional notes")
            btn=gr.Button("Save daily check-in",variant="primary"); msg=gr.Markdown(); daily=gr.Dataframe(label="Observation history")
            btn.click(daily_save,[state,b,d,f,v,n],[state,msg,daily])

        with gr.Tab("02 · Periodic measurements"):
            gr.Markdown("Log clinician-supervised or research measurements with units, eye/side, device and protocol.")
            a=gr.Dropdown(["Pre-flight baseline","Weekly research assessment","Monthly research assessment","Clinically indicated assessment","Post-flight follow-up"],value="Monthly research assessment",label="Assessment")
            dom=gr.Dropdown(["Ocular structure","Visual function","Fluid / vascular mechanism","Brain / neuro-ocular research"],value="Ocular structure",label="Domain")
            m=gr.Dropdown(["OCT — peripapillary retinal thickness / RNFL","OCT — optic disc edema","OCT — ganglion cell layer","OCT / fundoscopy — chorioretinal folds","MRI — posterior globe shape","Refraction","Visual acuity","Visual field","Doppler — internal jugular venous flow","CSF / intracranial fluid dynamics research","MRI — brain / fluid distribution","Other"],label="Measurement")
            with gr.Row():
                val=gr.Textbox(label="Value / finding"); u=gr.Textbox(label="Unit")
                side=gr.Dropdown(["Both eyes","Right eye","Left eye","Not applicable / unspecified"],value="Both eyes",label="Eye / side")
            protocol=gr.Textbox(label="Instrument / protocol"); note=gr.Textbox(label="Notes / source")
            rb=gr.Button("Log measurement",variant="primary"); rm=gr.Markdown(); rdf=gr.Dataframe(label="Research measurement history")
            rb.click(research_save,[state,a,dom,m,val,u,side,protocol,note],[state,rm,rdf])

        with gr.Tab("03 · Relationship explorer"):
            gr.Markdown("### Raw measurements — separate panels preserve original units")
            gr.Plot(value=timeline_raw())
            gr.Markdown("### Baseline-normalized comparison — each series starts at 100")
            gr.Plot(value=timeline_normalized())
            gr.Dataframe(value=normalized_table(),label="Normalized values (% of each measure's own baseline)")
            gr.Markdown("### Pairwise relationships")
            gr.Plot(value=correlation_matrix())
            with gr.Row():
                x=gr.Dropdown(NORM_COLS,value=NORM_COLS[1],label="Candidate mechanism (normalized)")
                y=gr.Dropdown(NORM_COLS,value=NORM_COLS[2],label="Ocular / functional outcome (normalized)")
            ex=gr.Button("Explore normalized relationship",variant="primary"); p=gr.Plot(); t=gr.Markdown()
            ex.click(relationship,[x,y],[p,t])
            gr.Markdown("### Exploratory time-lag comparison")
            lag=gr.Slider(0,30,value=15,step=15,label="Candidate mechanism lead time (mission days)")
            lagbtn=gr.Button("Explore lagged association"); lagout=gr.Markdown()
            lagbtn.click(lag_explorer,[x,y,lag],lagout)
            gr.Markdown("**Interpretation safeguards:** the example uses seven synthetic time points at 15-day intervals. Normalization does not make measurements biologically equivalent. Correlation and lagged association do not establish causation.")

        with gr.Tab("04 · CFD / FEA workbench"):
            gr.Markdown("Transparent mathematical teaching surrogates only—not full anatomical CFD or FEA solvers.")
            gr.Markdown("### CFD · rigid circular-tube flow sensitivity")
            with gr.Row():
                rad=gr.Slider(.2,3,value=1,step=.1,label="Tube radius (mm)")
                leng=gr.Slider(5,50,value=20,step=1,label="Tube length (mm)")
            with gr.Row():
                dp=gr.Slider(0,100,value=5,step=1,label="Pressure difference (Pa)")
                mu=gr.Slider(.5,5,value=1,step=.1,label="Dynamic viscosity (mPa·s)")
            cb=gr.Button("Calculate idealized flow",variant="primary"); co=gr.Markdown(); cb.click(cfd,[rad,leng,dp,mu],co)
            gr.Markdown("### FEA · one-dimensional elastic response proxy")
            with gr.Row():
                sig=gr.Slider(0,20,value=2,step=.5,label="Nominal stress (kPa)")
                young=gr.Slider(.05,5,value=1,step=.05,label="Young's modulus (MPa)")
                char=gr.Slider(.1,10,value=2,step=.1,label="Characteristic length (mm)")
            fb=gr.Button("Calculate elastic proxy",variant="primary"); fo=gr.Markdown(); fb.click(fea,[sig,young,char],fo)
            gr.Markdown("A genuine coupled model requires anatomical geometry, fluid and tissue properties, pulsatile/venous boundary conditions, constitutive laws, mesh and solver verification, sensitivity analysis and validation against ocular measurements.")

    gr.HTML("""
    <div class="nv-foot">
      NEUROVISION · SPACEFLIGHT VISION RESEARCH · CFD / FEA EXPLORATION · FUTURE SI-ASSISTED EVIDENCE REVIEW<br>
      Long-term ambition: contribute to optic nerve protection, regeneration research and vision restoration.
    </div>
    """)

if __name__=="__main__":
    app.launch(share=True)
